[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb) [![View in nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_proteomics.ipynb)

# Proteomics

Run brain, heart and kidney OrganAge clocks on real plasma measurements.

In [1]:
import pandas as pd
import pyaging as pya

/Users/lucascamillo/.codex/worktrees/4cd8/pyaging/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download and load example data

This [PAD000022](https://www.ebi.ac.uk/pride/archive/projects/PAD000022) subset contains 32 subjects and 134 Olink Explore 3072 protein assays from [Kirsher et al.](https://doi.org/10.1038/s42004-025-01665-1). All selected measurements pass sample and assay QC. The PRIDE data are CC0.

NPX values retain the source plate-control normalization and log2 scale, including negative values. No values were simulated, imputed or rescaled.

In [2]:
try:
    example_path = pya.data.download_example_data("PAD000022")
except ValueError:  # Compatibility with pyaging releases through 0.5.7.
    from huggingface_hub import hf_hub_download

    example_path = hf_hub_download("lucascamillomd/pyaging-data", "PAD000022_subset.pkl")

In [3]:
df = pd.read_pickle(example_path)

In [4]:
df.head()

,subject_id,ABCA2,ADAM22,ADCYAP1R1,AMBN,ANGPTL7,APLP1,APOD,ATP1B2,ATP6V1G2,...,TBR1,TMPRSS5,TNNI3,TNR,TSPAN7,TUBB3,UMOD,VSNL1,VSTM2B,VWC2L
SampleID,,,,,,,,,,,,,,,,,,,,,
227979487,5530212077,-1.0487,0.3775,-0.8507,-0.3239,-0.6959,2.1725,-0.4172,0.7162,-1.0262,...,0.1173,0.1000,0.3241,1.0794,-0.9204,-0.2996,-0.6762,-0.2694,-0.9294,-0.6177
227979500,5530211357,0.9520,0.1226,0.3178,-0.1060,0.5560,1.0653,-0.1388,0.9761,-0.3490,...,-0.1608,0.7520,0.9370,-0.4689,-0.3990,0.4868,-0.0448,-0.2960,-0.5715,-0.1843
227979563,5530212493,-0.8013,0.3672,-0.5698,0.3369,0.0453,0.4342,-0.5647,0.6476,-1.1413,...,-1.0652,0.1580,-6.8962,-0.6484,-1.9722,0.0274,-0.1437,-0.7229,-0.6959,-0.0661
227980531,5530212082,0.8554,0.3239,-0.2408,0.3130,-1.0838,1.9086,-0.5241,0.3991,-1.5048,...,0.4509,-0.5151,-0.4877,0.7595,0.7363,-0.0296,0.4967,-0.7202,-1.1552,-0.0851
227980548,5530212143,0.6599,-0.2073,-0.3333,0.4523,-0.2309,-0.2604,-0.7756,0.4986,-0.8632,...,-0.3154,-0.2997,-6.8859,-0.7075,0.1607,1.0697,-0.1327,0.2001,-1.1729,-0.2374


## Convert data to AnnData object

Keep `subject_id` as metadata and preserve the case-sensitive protein names. This subset has no missing values, so conversion needs no imputation.

In [5]:
adata = pya.pp.df_to_adata(df, metadata_cols=["subject_id"], verbose=False)

In [6]:
adata

AnnData object with n_obs × n_vars = 32 × 134
    obs: 'subject_id'
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

The chronological clocks return years. The brain mortality clock returns relative natural-log mortality hazard. These raw scores have not been calibrated to this cohort. Platform matching alone does not harmonize NPX between studies.

PAC, HPS and PAOPAC also require age in years, which this public dataset does not supply.

In [7]:
clock_names = [
    "organagechronologicalbrain",
    "organagechronologicalheart",
    "organagechronologicalkidney",
    "organagemortalitybrain",
]
pya.pred.predict_age(adata, clock_names, verbose=False)

In [8]:
adata.obs.head()

,subject_id,organagechronologicalbrain,organagechronologicalheart,organagechronologicalkidney,organagemortalitybrain
SampleID,,,,,
227979487,5530212077,47.857161,57.048704,53.750039,-1.895396
227979500,5530211357,75.595309,49.269629,56.792420,2.006320
227979563,5530212493,44.751195,53.019771,55.054821,-2.005122
227980531,5530212082,47.900347,56.762836,51.138736,-2.356227
227980548,5530212143,39.511782,53.157468,50.715471,-2.254959


All four clocks have complete feature coverage in this subset.

In [9]:
pd.Series(
    {name: adata.uns[f"{name}_percent_na"] for name in clock_names},
    name="missing_features_percent",
)

organagechronologicalbrain     0.0
organagechronologicalheart     0.0
organagechronologicalkidney    0.0
organagemortalitybrain         0.0
Name: missing_features_percent, dtype: float64

## Get citation

Clock citations, units and preparation notes are stored in `adata.uns`. Dataset provenance is in `df.attrs`; the [data card](https://huggingface.co/lucascamillomd/pyaging-data/blob/main/PAD000022_README.md) records the selection and source checksums.

In [10]:
adata.uns["organagechronologicalbrain_metadata"]

{'clock_name': 'organagechronologicalbrain',
 'data_type': 'proteomics',
 'species': 'Homo sapiens',
 'year': 2025,
 'approved_by_author': '⌛',
 'citation': 'Goeminne, L. J. E., et al. "Plasma protein-based organ-specific aging and mortality models unveil diseases as accelerated aging of organismal systems." Cell Metabolism 37, 205–222.e6 (2025).',
 'doi': 'https://doi.org/10.1016/j.cmet.2024.10.005',
 'notes': "Brain OrganAge chronological model for Olink Explore 3072, using author-recommended fold 1. Trained on UK Biobank plasma measured with Olink Explore 3072. Input columns are original case-sensitive protein symbols (e.g. NTproBNP, HLA-DRA), not UniProt IDs or Olink assay IDs. Values are assay-normalized Olink NPX on the log2 relative-abundance scale, not raw counts, concentrations, linear abundances or z-scores. No additional centering or cohort scaling is applied. Absent proteins contribute zero; NaNs in supplied proteins propagate, matching the original Example_Script.R scoring

In [11]:
df.attrs

{'accession': 'PAD000022',
 'source_doi': '10.6019/PAD000022',
 'publication_doi': '10.1038/s42004-025-01665-1',
 'license': 'CC0-1.0',
 'platform': 'Olink Explore 3072',
 'unit': 'NPX',
 'normalization': 'Plate control, unchanged from the source CSV'}